# 🚀 ProdiBot — Personal Productivity Assistant## Final Project: LLM-Based Tools and Gemini API Integration for Data Scientists### Deskripsi ProyekProdiBot adalah chatbot **Personal Productivity Assistant** berbasis AI yang dibangun menggunakan **Google Gemini API** dan di-deploy menggunakan **Streamlit**. Chatbot ini dilengkapi dengan tiga fitur utama:1. **🤖 LLM Chat** — Percakapan natural menggunakan Gemini dengan system instruction khusus produktivitas2. **🔧 Function Calling** — 9 tools untuk manajemen tugas, jadwal, ringkasan harian, dan tips produktivitas3. **📄 RAG (Retrieval-Augmented Generation)** — Upload dokumen PDF dan tanya jawab berdasarkan isinya### Arsitektur Sistem```User Input (Streamlit Chat)         │         ├── Pertanyaan umum ──────────────► Gemini LLM ──► Response         │         ├── Perintah tugas/jadwal ─────────► Function Calling         │                                      │         │                                      ├── add_task / list_tasks / complete_task / delete_task         │                                      ├── add_schedule / list_schedules / delete_schedule         │                                      ├── get_daily_summary         │                                      └── get_productivity_tip         │         └── Pertanyaan tentang dokumen ───► RAG Pipeline                                               │                                               ├── PDF → PyPDFLoader                                               ├── Chunking → RecursiveCharacterTextSplitter                                               ├── Embedding → GoogleGenerativeAIEmbeddings                                               ├── Vector Store → ChromaDB                                               └── Retrieval → Top-K similarity search```### Struktur File Proyek```Final Project/├── chatbot_project.ipynb      ← Notebook dokumentasi proyek (file ini)├── streamlit_chatbot.py       ← Streamlit app untuk deployment├── requirements.txt           ← Daftar dependencies├── .env                       ← API Key (GEMINI_API_KEY)├── .streamlit/│   └── config.toml            ← Konfigurasi tema Streamlit└── utils/    ├── __init__.py    ├── prompts.py              ← System instruction & prompt templates    ├── tools.py                ← Function calling tools (9 tools)    ├── rag.py                  ← RAG pipeline (PDF → ChromaDB → Retrieval)    └── llm.py                  ← Gemini client & chat session```

---# Section 1: Setup & InstalasiInstall semua dependencies yang dibutuhkan. Library utama:- `google-genai` — SDK Google Gemini API- `streamlit` — Framework web app untuk deployment- `langchain` & `langchain-google-genai` — Framework RAG + integrasi Gemini- `chromadb` — Vector database untuk menyimpan embedding dokumen- `pypdf` — Membaca file PDF

In [ ]:
# Install dependencies!pip install -q streamlit python-dotenv google-genai langchain langchain-google-genai langchain-community langchain-text-splitters chromadb pypdf

In [ ]:
# Import library utamaimport osimport warningswarnings.filterwarnings('ignore')from dotenv import load_dotenvfrom google import genaifrom google.genai import typesfrom IPython.display import Markdown as md# Load API key dari .envload_dotenv()api_key = os.getenv("GEMINI_API_KEY")# Verifikasi API keyif api_key:    print(f"✅ API Key berhasil dimuat (***{api_key[-4:]})")else:    print("⚠️ API Key tidak ditemukan. Masukkan secara manual:")    api_key = input("Masukkan GEMINI_API_KEY: ")

---# Section 2: System Prompt & PersonaSystem instruction mendefinisikan **persona** dan **aturan perilaku** chatbot. ProdiBot dikonfigurasi sebagai asisten produktivitas yang ramah, menggunakan Bahasa Indonesia, dan proaktif memberikan motivasi.### Komponen Prompt:1. **SYSTEM_INSTRUCTION** — Persona, kemampuan, dan aturan perilaku2. **RAG_CONTEXT_TEMPLATE** — Template untuk menyisipkan konteks dokumen3. **WELCOME_MESSAGE** — Pesan sambutan saat pertama kali membuka chatbot

In [ ]:
# Tampilkan System Instruction dari utils/prompts.pyfrom utils.prompts import SYSTEM_INSTRUCTION, RAG_CONTEXT_TEMPLATE, WELCOME_MESSAGEprint("=" * 70)print("SYSTEM INSTRUCTION")print("=" * 70)print(SYSTEM_INSTRUCTION)

In [ ]:
# Tampilkan RAG Context Templateprint("=" * 70)print("RAG CONTEXT TEMPLATE")print("=" * 70)print(RAG_CONTEXT_TEMPLATE)

In [ ]:
# Tampilkan Welcome Messageprint("=" * 70)print("WELCOME MESSAGE")print("=" * 70)print(WELCOME_MESSAGE)

---# Section 3: Function Calling Tools**Function Calling** memungkinkan model Gemini memanggil fungsi Python secara otomatis berdasarkan permintaan user. ProdiBot memiliki **9 tools** yang terbagi dalam 4 kategori:| Kategori | Tools | Deskripsi ||---|---|---|| 📝 Task Management | `add_task`, `list_tasks`, `complete_task`, `delete_task` | CRUD tugas dengan prioritas dan deadline || 📅 Schedule Management | `add_schedule`, `list_schedules`, `delete_schedule` | CRUD jadwal/acara || 📊 Daily Summary | `get_daily_summary` | Ringkasan tugas pending & jadwal hari ini || 💡 Productivity Tips | `get_productivity_tip` | Tips produktivitas acak |### Cara Kerja Function Calling di Gemini```User: "Tambah tugas belajar Python deadline besok"         ↓Gemini menganalisis → memutuskan memanggil add_task()         ↓SDK auto-execute: add_task(task="belajar Python", priority="medium", due_date="2026-05-06")         ↓Hasil dikembalikan ke Gemini → Gemini format jawaban untuk user```> **Note**: ProdiBot menggunakan **Automatic Function Calling** — SDK Gemini otomatis mengeksekusi tool tanpa perlu loop manual.

In [ ]:
# Tampilkan semua tools yang tersediafrom utils.tools import get_all_toolstools = get_all_tools()print(f"📦 Total tools yang terdaftar: {len(tools)}\n")for i, tool in enumerate(tools, 1):    print(f"  {i}. {tool.__name__}")    # Tampilkan docstring singkat (baris pertama)    doc = tool.__doc__.strip().split("\n")[0] if tool.__doc__ else "No description"    print(f"     └─ {doc}")    print()

### Kode Tools — Task ManagementBerikut contoh implementasi tool `add_task` dan `list_tasks`. Setiap tool adalah **fungsi Python biasa** dengan docstring yang jelas — Gemini membaca docstring ini untuk memahami kapan dan bagaimana memanggil tool.

In [ ]:
# Tampilkan source code dari tool add_taskimport inspectfrom utils.tools import add_task, list_tasksprint("=" * 70)print("TOOL: add_task")print("=" * 70)print(inspect.getsource(add_task))

In [ ]:
# Tampilkan source code dari tool list_tasksprint("=" * 70)print("TOOL: list_tasks")print("=" * 70)print(inspect.getsource(list_tasks))

### Demo Function CallingMari kita uji coba function calling secara langsung dengan mengirim pesan ke Gemini yang memicu pemanggilan tools.

In [ ]:
# Setup client dan chat session dengan toolsfrom utils.llm import create_client, create_chat_session, send_messageclient = create_client(api_key)chat = create_chat_session(client, model_id="gemini-2.5-flash", temperature=0.3)# Test 1: Tambah tugas — seharusnya memanggil add_task()response = send_message(chat, "Tambahkan tugas belajar Python dengan prioritas high dan deadline 2026-05-10")print("📩 Response:")print(response)

In [ ]:
# Test 2: Lihat daftar tugas — seharusnya memanggil list_tasks()response = send_message(chat, "Tampilkan semua tugas saya")print("📩 Response:")print(response)

In [ ]:
# Test 3: Tips produktivitas — seharusnya memanggil get_productivity_tip()response = send_message(chat, "Berikan saya tips produktivitas")print("📩 Response:")print(response)

---# Section 4: RAG Pipeline (Retrieval-Augmented Generation)RAG memungkinkan chatbot menjawab pertanyaan berdasarkan **dokumen yang di-upload user**. Pipeline RAG pada ProdiBot:```PDF Upload    ↓PyPDFLoader          ← Baca halaman per halaman    ↓RecursiveCharacter   ← Potong jadi chunks (500 karakter, overlap 100)TextSplitter    ↓GoogleGenerativeAI   ← Ubah teks jadi vector (embedding)Embeddings    ↓ChromaDB             ← Simpan vector di database (in-memory)    ↓User bertanya    ↓Retriever            ← Cari top-5 chunks paling relevan    ↓RAG_CONTEXT_TEMPLATE ← Gabungkan konteks + pertanyaan    ↓Gemini (LLM)         ← Generate jawaban berdasarkan konteks    ↓Jawaban```### Komponen Utama RAG di `utils/rag.py`:1. **`process_pdf()`** — Proses lengkap: baca PDF → chunk → embed → simpan ke ChromaDB2. **`query_documents()`** — Cari chunks relevan dari ChromaDB3. **`format_docs()`** — Gabungkan list dokumen jadi string konteks4. **`is_rag_query()`** — Heuristik untuk mendeteksi apakah pertanyaan user tentang dokumen

In [ ]:
# Tampilkan source code RAG pipelineimport inspectfrom utils.rag import process_pdf, query_documents, format_docs, is_rag_queryprint("=" * 70)print("FUNGSI: process_pdf")print("=" * 70)print(inspect.getsource(process_pdf))

In [ ]:
# Tampilkan fungsi query dan formatprint("=" * 70)print("FUNGSI: query_documents")print("=" * 70)print(inspect.getsource(query_documents))print("\n" + "=" * 70)print("FUNGSI: format_docs")print("=" * 70)print(inspect.getsource(format_docs))

### Demo RAG PipelineKita akan mendownload contoh PDF dan menguji RAG pipeline secara langsung di notebook.

In [ ]:
# Download contoh PDF untuk demo RAGimport urllib.requestpdf_url = "https://www.pearsonvue.com/content/dam/VUE/vue/en/documents/clients/it-specialist/its-od-307-artificial-intel-pearson.pdf"pdf_path = "demo_ai_cert.pdf"urllib.request.urlretrieve(pdf_url, pdf_path)print(f"✅ PDF berhasil didownload: {pdf_path}")

In [ ]:
# Proses PDF melalui RAG pipelinefrom langchain_community.document_loaders import PyPDFLoaderfrom langchain_text_splitters import RecursiveCharacterTextSplitterfrom langchain_google_genai import GoogleGenerativeAIEmbeddingsfrom langchain_community.vectorstores import Chroma# 1. Load PDFloader = PyPDFLoader(pdf_path)pages = loader.load_and_split()print(f"📄 Jumlah halaman: {len(pages)}")# 2. Chunkingtext_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)chunks = text_splitter.split_documents(pages)print(f"🧩 Jumlah chunks: {len(chunks)}")# 3. Embedding + ChromaDBembedding_model = GoogleGenerativeAIEmbeddings(    google_api_key=api_key,    model="models/text-embedding-004",)vector_store = Chroma.from_documents(documents=chunks, embedding=embedding_model)retriever = vector_store.as_retriever(search_kwargs={"k": 5})print(f"✅ Vector store berhasil dibuat!")

In [ ]:
# 4. Query dokumen — cari chunks relevanfrom utils.rag import format_docsquestion = "Apa saja topik utama dalam sertifikasi AI dari Pearson VUE?"docs = retriever.invoke(question)print(f"🔍 Pertanyaan: {question}")print(f"📚 Jumlah chunks relevan: {len(docs)}")print(f"\n--- Chunk pertama ---")print(docs[0].page_content[:300])

In [ ]:
# 5. Kirim pertanyaan + konteks ke Gemini (RAG)from utils.llm import send_message_with_ragrag_context = format_docs(docs)chat_rag = create_chat_session(client, model_id="gemini-2.5-flash", temperature=0.3)response = send_message_with_rag(chat_rag, question, rag_context)print("📩 Jawaban RAG:")print(response)

---# Section 5: LLM Client & Chat SessionModule `utils/llm.py` menangani inisialisasi Gemini client dan manajemen chat session. Menggunakan SDK `google-genai` (bukan langchain).### Fungsi Utama:| Fungsi | Deskripsi ||---|---|| `create_client(api_key)` | Membuat Gemini client baru || `create_chat_session(client, model_id, temperature)` | Membuat chat session dengan system instruction + tools || `send_message(chat, message)` | Kirim pesan (dengan auto function calling) || `send_message_with_rag(chat, message, rag_context)` | Kirim pesan + konteks RAG |

In [ ]:
# Tampilkan source code LLM moduleimport inspectfrom utils.llm import create_client, create_chat_session, send_message, send_message_with_ragprint("=" * 70)print("MODULE: utils/llm.py")print("=" * 70)for func in [create_client, create_chat_session, send_message, send_message_with_rag]:    print(f"\n{'─' * 50}")    print(f"FUNGSI: {func.__name__}")    print(f"{'─' * 50}")    print(inspect.getsource(func))

---# Section 6: Streamlit App — Deployment CodeBerikut adalah kode lengkap `streamlit_chatbot.py` yang mengintegrasikan semua komponen ProdiBot.### Fitur UI Streamlit:- **Sidebar**: API key, model selection, temperature, PDF upload, reset, task/schedule display- **Chat Interface**: `st.chat_message()` + `st.chat_input()`- **Session State**: Menyimpan riwayat chat, tugas, jadwal, dan RAG retriever- **Auto Function Calling**: Gemini otomatis memanggil tools saat user meminta- **RAG Integration**: Upload PDF → tanya jawab berdasarkan isi dokumen

In [ ]:
# Tampilkan isi file streamlit_chatbot.py# File ini sudah dibuat sebelumnya, kita tampilkan isinyawith open("streamlit_chatbot.py", "r", encoding="utf-8") as f:    content = f.read()print(f"✅ File streamlit_chatbot.py ({len(content)} karakter)")print(f"   Jumlah baris: {len(content.splitlines())}")print()print("=" * 70)print("PREVIEW (50 baris pertama):")print("=" * 70)for i, line in enumerate(content.splitlines()[:50], 1):    print(f"{i:3d} | {line}")

### Penjelasan Alur Streamlit App```App dijalankan (streamlit run streamlit_chatbot.py)         ↓1. Load .env & konfigurasi halaman         ↓2. Sidebar: API key, model, temperature, PDF upload         ↓3. Validasi API key → st.stop() jika kosong         ↓4. Inisialisasi Gemini client & chat session   (hanya jika belum ada di session_state atau config berubah)         ↓5. Tampilkan welcome message & riwayat chat         ↓6. User kirim pesan via st.chat_input()         ↓7. Cek: apakah pertanyaan tentang dokumen? (is_rag_query)   ├── Ya → query_documents() → send_message_with_rag()   └── Tidak → send_message() (dengan auto function calling)         ↓8. Tampilkan respons & simpan ke riwayat         ↓9. st.rerun() untuk update sidebar (tugas/jadwal baru)```

---# Section 7: Menjalankan Streamlit## Opsi A: Menjalankan di Lokal```bash# 1. Install dependenciespip install -r requirements.txt# 2. Pastikan file .env berisi GEMINI_API_KEY# 3. Jalankan Streamlitstreamlit run streamlit_chatbot.py```## Opsi B: Menjalankan di Google ColabGunakan **ngrok** untuk membuat tunnel publik ke Streamlit yang berjalan di Colab.

In [ ]:
# === UNTUK GOOGLE COLAB ===# Uncomment cell di bawah untuk menjalankan di Colab# !pip install -q pyngrok# from pyngrok import ngrok# from google.colab import userdata# import subprocess, time# # Set ngrok auth token (simpan di Colab Secrets)# ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))# def run_streamlit(filename, port=8501):#     subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)#     subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)#     ngrok.kill()#     time.sleep(3)#     proc = subprocess.Popen(#         ["streamlit", "run", filename,#          "--server.headless=true",#          "--server.port", str(port),#          "--server.enableCORS=false"],#         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL#     )#     time.sleep(3)#     public_url = ngrok.connect(port)#     print(f"🚀 ProdiBot berjalan di: {public_url}")#     return proc# proc = run_streamlit("streamlit_chatbot.py")

---# Section 8: Kesimpulan## Ringkasan Proyek**ProdiBot** adalah chatbot Personal Productivity Assistant yang menggabungkan tiga teknologi utama:| Fitur | Teknologi | Implementasi ||---|---|---|| 🤖 LLM Chat | Google Gemini API (`google-genai`) | `utils/llm.py` || 🔧 Function Calling | Gemini Automatic Function Calling | `utils/tools.py` (9 tools) || 📄 RAG | LangChain + ChromaDB + Gemini Embeddings | `utils/rag.py` || 🎨 Deployment | Streamlit | `streamlit_chatbot.py` |## Parameter Kreatif yang Digunakan- **Gaya bahasa**: Profesional namun ramah (Bahasa Indonesia)- **Domain**: Produktivitas pribadi (task management, scheduling)- **Fitur tambahan**: RAG untuk tanya jawab dokumen, motivasi & tips produktivitas- **UI/UX**: Dark theme dengan gradient sidebar, emoji, dan status cards## Deliverables1. ✅ **Notebook dokumentasi** — `chatbot_project.ipynb`2. ✅ **Streamlit app** — `streamlit_chatbot.py`3. ✅ **Utility modules** — `utils/prompts.py`, `utils/llm.py`, `utils/tools.py`, `utils/rag.py`4. ✅ **Konfigurasi** — `requirements.txt`, `.env`, `.streamlit/config.toml`